**Simulation 1**

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import Checkbox, HBox, VBox, Output, Label
from IPython.display import display

#basic parameters
T = 1/220
f0 = 1 / T
t = np.linspace(0, 2*T, 1000)
t_ms = t * 1000

# square wave
def square_wave(t, T):
    return np.where((t % T) < (T/2), 1, -1)

f_square = square_wave(t, T)

# harmonic component k
def harmonic_component(t, k, f0):
    omega0 = 2 * np.pi * f0
    return (4/np.pi) * (1/k) * np.sin(k * omega0 * t)


harmonics = [1, 3, 5, 7, 9, 11, 13, 15, 17, 19]  # list of harmonics. Can be added more
checkboxes = [
    Checkbox(value=(k <= 5), description=f'k = {k}', indent=False)
    for k in harmonics
]

output = Output()

def update_plot(change=None):
    with output:
        output.clear_output(wait=True)

        plt.figure(figsize=(10, 6))


        plt.plot(t_ms, f_square, 'k--', label='Original Waveform')


        total = np.zeros_like(t)
        any_checked = False


        for cb, k in zip(checkboxes, harmonics):
            if cb.value:
                any_checked = True
                comp = harmonic_component(t, k, f0)
                total += comp
                plt.plot(t_ms, comp, linestyle=':', alpha=0.6, label=f'Harmonic {k}')


        if any_checked:
            plt.plot(t_ms, total, linewidth=2, label='Harmonics Superposition')

        plt.xlabel("Time (ms)")
        plt.ylabel("Amplitude")
        plt.ylim(-1.8, 1.8)
        plt.legend(loc='upper right', fontsize=8)
        plt.show()

for cb in checkboxes:
    cb.observe(update_plot, 'value')

controls = HBox(checkboxes)
display(VBox([Label("Select harmonics to display:"), controls, output]))

update_plot()


**Simulation 2**

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

# basic parameters
T = 1/220
f0 = 1 / T
t = np.linspace(0, 2*T, 1000)
t_ms = t * 1000

# square wave
def square_wave(t, T):
    return np.where((t % T) < (T/2), 1, -1)

f_square = square_wave(t, T)

#square wave approximation with fourier series
def square_wave_approx(t, N, f0):
    """
    Aproksimasi gelombang kotak dengan deret Fourier.
    N = jumlah harmonik ganjil maksimum (1,3,5,...)
    """
    omega0 = 2 * np.pi * f0
    s = np.zeros_like(t)


    for k in range(1, N+1, 2):
        s += (4/np.pi) * (1/k) * np.sin(k * omega0 * t)
    return s

def plot_fourier(N):
    plt.figure(figsize=(10, 5))


    plt.plot(t_ms, f_square, 'k--', label='Original waveform')


    s_t = square_wave_approx(t, N, f0)
    plt.plot(t_ms, s_t, label=f"Approximations {N} harmonics")

    plt.xlabel("Time (ms)")
    plt.ylabel("Amplitude")
    plt.ylim(-1.5, 1.5)
    plt.legend()
    plt.show()


interact(
    plot_fourier,
    N=IntSlider(
        min=1,
        max=35,
        step=2,
        value=1,
        description="Harmonics maks"
    )
);


interactive(children=(IntSlider(value=1, description='Harmonics maks', max=35, min=1, step=2), Output()), _dom…

**Simulation 3**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# basic parameters
f0 = 440
T = 1 / f0
fs = 44100
duration = 5 * T
t = np.arange(0, duration, 1/fs)

# time-domain function
def square_wave_approx(t, N, f0):
    s_t = np.zeros_like(t)
    for k in range(1, 2*N, 2):
        s_t += (1/k) * np.sin(2*np.pi*k*f0*t)
    return (4/np.pi) * s_t


N_harm = 5
s_t = square_wave_approx(t, N_harm, f0)

#discrete fourier transform process
def dft(x):
    N = len(x)
    X = np.zeros(N, dtype=complex)
    for k in range(N):
        for n in range(N):
            X[k] += x[n] * np.exp(-2j * np.pi * k * n / N)
    return X

N = len(s_t)
S_f = dft(s_t) / N
freqs = np.arange(N) * fs / N


mask = freqs >= 0
freqs_pos = freqs[mask]
S_pos = np.abs(S_f[mask])


plt.figure(figsize=(12,5))
plt.plot(t*500, s_t)
plt.xlabel("Time (ms)")
plt.ylabel("Amplitude")
plt.title("Waveform in Time Domain (Approximated with 5 harmonics)")
plt.show()

# plot frequency domain
plt.figure(figsize=(12,5))
plt.stem(freqs_pos, S_pos, basefmt=" ")
plt.xlim(0,   4000)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude")
plt.title("Waveform in Frequency Domain")
#plt.grid(True)
plt.show()

**Simulation 4**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# reading data from time-series-all.txt
data = np.loadtxt("time-series-all.txt")
t = data[:, 0]
x = data[:, 1]


plt.figure(figsize=(10,4))
plt.plot(t, x)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.title("Time Domain Voice Signal")
plt.show()


N = len(x)
dt = t[1] - t[0]
fs = 1/dt

X = np.fft.fft(x) / N
freqs = np.fft.fftfreq(N, d=dt)


half = N//2
plt.figure(figsize=(10,4))
plt.stem(freqs[:half], np.abs(X[:half]), basefmt=" ")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude")
plt.title("The Fourier-Transformed Voice Signal")
#plt.grid(True)
plt.show()